# Attention-Guided Swin Transformer for Image Compression
## MSc Dissertation 

This notebook covers the training, validation, and Rate-Distortion (R-D) curve plotting for the custom Attention-Guided Swin Transformer over the Honeybee UVG 512x512 dataset.

In [ ]:
!pip install compressai timm pytorch-msssim matplotlib

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from compressai.losses import RateDistortionLoss
from model import AttentionGuidedSwinCompression

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Kaggle Dataset Path - Based on your specific path
DATASET_PATH = '/kaggle/input/datasets/jeevajoji/uvg-honeybee-512x512/honeybee_512_crop/'

### Custom PyTorch Dataset
Since the images are in a flat directory without class folders, we use a custom Dataset class.

In [ ]:
class UVGDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        # Grab all PNG files in the directory
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '*.png')))
        self.transform = transform
        if len(self.image_paths) == 0:
            print(f"Warning: No PNG images found in {root_dir}")
        else:
            print(f"Found {len(self.image_paths)} images.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Convert to RGB to ensure 3 channels
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, 0  # Return dummy label 0

transform = transforms.Compose([
    transforms.RandomCrop(256), # Train on 256x256 crops to save VRAM and increase batches
    transforms.ToTensor()
])

train_dataset = UVGDataset(DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)


### Training Loop for Multiple Lambdas (R-D Curve generation)

In [ ]:
def train_model_for_lambda(lmbda, epochs=EPOCHS):
    print(f"\n--- Training model for Lambda = {lmbda} ---")
    model = AttentionGuidedSwinCompression(N=128, M=192).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = RateDistortionLoss(lmbda=lmbda)
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_bpp = 0.0
        epoch_mse = 0.0
        
        for i, (images, _) in enumerate(train_loader):
            images = images.to(DEVICE)
            optimizer.zero_grad()
            
            out_net = model(images)
            out_criterion = criterion(out_net, images)
            
            out_criterion["loss"].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += out_criterion["loss"].item()
            epoch_bpp += out_criterion["bpp_loss"].item()
            epoch_mse += out_criterion["mse_loss"].item()
            
        avg_loss = epoch_loss / len(train_loader)
        avg_bpp = epoch_bpp / len(train_loader)
        avg_mse = epoch_mse / len(train_loader)
        print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | BPP: {avg_bpp:.4f} | MSE: {avg_mse:.6f}")
        
    # Save the model weights specific to this lambda
    torch.save(model.state_dict(), f"swin_compress_lambda_{lmbda}.pth")
    return avg_bpp, avg_mse

In [ ]:
# Define the lambdas for different quality levels (Low quality to High quality)
lambdas = [0.0018, 0.0035, 0.0067, 0.0130, 0.0250]

rd_results = {"bpp": [], "mse": []}

# --- UNCOMMENT TO RUN FULL TRAINING SUITE ---
# for l in lambdas:
#     final_bpp, final_mse = train_model_for_lambda(l, epochs=EPOCHS)
#     rd_results["bpp"].append(final_bpp)
#     rd_results["mse"].append(final_mse)


### Plotting the Rate-Distortion Curve

In [ ]:
# --- UNCOMMENT TO PLOT AFTER TRAINING ---
# import math
# psnr_values = [10 * math.log10(1.0 / mse) for mse in rd_results["mse"]]

# plt.figure(figsize=(8, 6))
# plt.plot(rd_results["bpp"], psnr_values, marker='o', linestyle='-', color='b', label='Attention-Guided Swin')
# plt.title('Rate-Distortion Curve (Honeybee UVG 512x512)')
# plt.xlabel('Bitrate (BPP - Bits Per Pixel)')
# plt.ylabel('PSNR (dB)')
# plt.grid(True)
# plt.legend()
# plt.show()
